# EchoTrace - Fake Review Detection Model Training

## 🎯 Overview
Train a machine learning ensemble model to detect fake reviews using:
- TF-IDF Vectorization
- 4-Model Soft Voting Ensemble
- Cross-Validation

## 📋 Instructions
1. Upload your `synthetic_reviews.csv` file
2. Run all cells in order
3. Download the trained model files at the end

## 1️⃣ Install & Import Libraries

In [ ]:
# Install required packages (if needed)
!pip install -q scikit-learn pandas numpy matplotlib seaborn

import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, VotingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score, 
                             f1_score, roc_auc_score, roc_curve, precision_recall_curve)

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

print("✅ All libraries imported successfully!")
print(f"⏰ Training started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2️⃣ Upload Dataset

Click the "Choose Files" button below and upload your `synthetic_reviews.csv` file

In [ ]:
from google.colab import files

print("📤 Please upload your synthetic_reviews.csv file:")
uploaded = files.upload()

if 'synthetic_reviews.csv' in uploaded:
    print("\n✅ File uploaded successfully!")
else:
    print("\n⚠️  Warning: synthetic_reviews.csv not found. Please upload the correct file.")

## 3️⃣ Load & Explore Dataset

In [ ]:
# Load data
df = pd.read_csv('synthetic_reviews.csv', encoding='utf-8')

print("="*70)
print("DATASET STATISTICS")
print("="*70)
print(f"Total samples: {len(df)}")
print(f"\nColumns: {df.columns.tolist()}")

# Display class distribution
print(f"\nClass Distribution:")
print(df['label'].value_counts())
print(f"\nPercentages:")
print(df['label'].value_counts(normalize=True) * 100)

# Show first few rows
print("\n📋 First 3 rows:")
df.head(3)

In [ ]:
# Extract features and labels
texts = df['review_text'].tolist()
labels = df['label'].tolist()

print("📝 Sample genuine review (label=0):")
genuine_idx = labels.index(0)
print(f"   {texts[genuine_idx][:200]}...\n")

print("⚠️  Sample fake review (label=1):")
fake_idx = labels.index(1)
print(f"   {texts[fake_idx][:200]}...")

## 4️⃣ Text Vectorization (TF-IDF)

In [ ]:
print("="*70)
print("TEXT VECTORIZATION (TF-IDF)")
print("="*70)

vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True,
    strip_accents='unicode',
    lowercase=True
)

X = vectorizer.fit_transform(texts)
y = np.array(labels)

print(f"✅ Vectorization complete!")
print(f"   Feature matrix shape: {X.shape}")
print(f"   Vocabulary size: {len(vectorizer.vocabulary_)}")
print(f"   Sparse matrix density: {(X.nnz / (X.shape[0] * X.shape[1]) * 100):.2f}%")

## 5️⃣ Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("="*70)
print("TRAIN-TEST SPLIT")
print("="*70)
print(f"Training samples: {len(y_train)} ({len(y_train)/len(y)*100:.1f}%)")
print(f"Test samples: {len(y_test)} ({len(y_test)/len(y)*100:.1f}%)")
print(f"\nTrain distribution: {dict(zip(*np.unique(y_train, return_counts=True)))}")
print(f"Test distribution: {dict(zip(*np.unique(y_test, return_counts=True)))}")

## 6️⃣ Build & Train Ensemble Model

In [ ]:
print("="*70)
print("BUILDING 4-MODEL ENSEMBLE")
print("="*70)
print("Models:")
print("  1️⃣  Logistic Regression")
print("  2️⃣  Calibrated Linear SVC")
print("  3️⃣  Gradient Boosting Classifier")
print("  4️⃣  Random Forest Classifier")
print("\nVoting Strategy: Soft (average probabilities)\n")

# Define base models
lr = LogisticRegression(
    C=1.0, penalty='l2', solver='lbfgs',
    class_weight='balanced', max_iter=500, random_state=42
)

svc_base = LinearSVC(
    C=1.0, penalty='l2', loss='squared_hinge',
    class_weight='balanced', max_iter=2000, random_state=42
)
svc = CalibratedClassifierCV(svc_base, cv=3)

gbc = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=5,
    min_samples_split=20, min_samples_leaf=10,
    subsample=0.8, random_state=42
)

rfc = RandomForestClassifier(
    n_estimators=100, max_depth=20,
    min_samples_split=10, min_samples_leaf=5,
    class_weight='balanced', random_state=42, n_jobs=-1
)

ensemble = VotingClassifier(
    estimators=[('lr', lr), ('svc', svc), ('gbc', gbc), ('rfc', rfc)],
    voting='soft',
    n_jobs=-1
)

print("⏳ Training in progress (this may take 1-3 minutes)...\n")
ensemble.fit(X_train, y_train)
print("✅ Training complete!")

## 7️⃣ Evaluate Model Performance

In [ ]:
# Make predictions
y_pred = ensemble.predict(X_test)
y_pred_proba = ensemble.predict_proba(X_test)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("="*70)
print("MODEL PERFORMANCE ON TEST SET")
print("="*70)
print(f"\n📊 Overall Metrics:")
print(f"   Accuracy:  {accuracy*100:.2f}%")
print(f"   F1-Score:  {f1:.4f}")
print(f"   ROC-AUC:   {roc_auc:.4f}")

print(f"\n📋 Detailed Classification Report:")
print(classification_report(y_test, y_pred, 
                          target_names=['Genuine (0)', 'Fake (1)'],
                          digits=4))

## 8️⃣ Cross-Validation

In [ ]:
print("⏳ Running 5-Fold Cross-Validation (this may take 2-5 minutes)...\n")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(ensemble, X_train, y_train, cv=skf, scoring='accuracy', n_jobs=-1)

print("="*70)
print("CROSS-VALIDATION RESULTS")
print("="*70)
for i, score in enumerate(cv_scores, 1):
    print(f"Fold {i}: {score*100:.2f}%")
print(f"\n📊 Mean CV Accuracy: {cv_scores.mean()*100:.2f}% (±{cv_scores.std()*100:.2f}%)")

## 9️⃣ Visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('EchoTrace Fake Review Detection - Performance Analysis', 
             fontsize=16, fontweight='bold', y=1.00)

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Genuine', 'Fake'],
            yticklabels=['Genuine', 'Fake'],
            cbar_kws={'label': 'Count'})
axes[0, 0].set_title('Confusion Matrix', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0, 1].plot(fpr, tpr, color='darkorange', lw=2, 
               label=f'ROC (AUC = {roc_auc:.4f})')
axes[0, 1].plot([0, 1], [0, 1], color='navy', lw=2, 
               linestyle='--', label='Random')
axes[0, 1].set_xlim([0.0, 1.0])
axes[0, 1].set_ylim([0.0, 1.05])
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve', fontsize=12, fontweight='bold')
axes[0, 1].legend(loc="lower right")
axes[0, 1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1, 0].plot(recall, precision, color='green', lw=2)
axes[1, 0].fill_between(recall, precision, alpha=0.2, color='green')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-Recall Curve', fontsize=12, fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].set_xlim([0.0, 1.0])
axes[1, 0].set_ylim([0.0, 1.05])

# 4. Cross-Validation Scores
colors = ['steelblue' if s >= cv_scores.mean() else 'lightcoral' for s in cv_scores]
axes[1, 1].bar(range(1, 6), cv_scores, color=colors, alpha=0.7, edgecolor='black')
axes[1, 1].axhline(y=cv_scores.mean(), color='red', linestyle='--', linewidth=2,
                   label=f'Mean: {cv_scores.mean()*100:.2f}%')
axes[1, 1].set_xlabel('Fold Number')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_title('5-Fold Cross-Validation', fontsize=12, fontweight='bold')
axes[1, 1].set_ylim([0.75, 1.0])
axes[1, 1].set_xticks(range(1, 6))
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print("✅ Visualizations generated!")

## 🔟 Save Trained Model

In [ ]:
print("="*70)
print("SAVING TRAINED MODEL")
print("="*70)

# Save ensemble model
model_filename = 'fake_review_model.pkl'
with open(model_filename, 'wb') as f:
    pickle.dump(ensemble, f)
print(f"✅ Model saved: {model_filename}")

# Save vectorizer
vectorizer_filename = 'tfidf_vectorizer.pkl'
with open(vectorizer_filename, 'wb') as f:
    pickle.dump(vectorizer, f)
print(f"✅ Vectorizer saved: {vectorizer_filename}")

# Save metadata
metadata = {
    'train_date': datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'total_samples': len(texts),
    'train_samples': len(y_train),
    'test_samples': len(y_test),
    'accuracy': float(accuracy),
    'f1_score': float(f1),
    'roc_auc': float(roc_auc),
    'cv_mean': float(cv_scores.mean()),
    'cv_std': float(cv_scores.std()),
    'features': X.shape[1],
    'vocabulary_size': len(vectorizer.vocabulary_)
}

metadata_filename = 'model_metadata.pkl'
with open(metadata_filename, 'wb') as f:
    pickle.dump(metadata, f)
print(f"✅ Metadata saved: {metadata_filename}")

print(f"\n📊 Training Summary:")
print(f"   Trained on: {len(y_train)} samples")
print(f"   Test Accuracy: {accuracy*100:.2f}%")
print(f"   CV Accuracy: {cv_scores.mean()*100:.2f}% ± {cv_scores.std()*100:.2f}%")

## 1️⃣1️⃣ Test Sample Predictions

In [ ]:
def test_review(review_text):
    """Test the model with a sample review"""
    review_vec = vectorizer.transform([review_text])
    prediction = ensemble.predict(review_vec)[0]
    probability = ensemble.predict_proba(review_vec)[0]
    
    label = "🚨 FAKE" if prediction == 1 else "✅ GENUINE"
    confidence = probability[prediction] * 100
    
    print(f"\n{'='*70}")
    print(f"📝 Review: {review_text[:120]}...")
    print(f"{'='*70}")
    print(f"🎯 Prediction: {label}")
    print(f"📊 Confidence: {confidence:.2f}%")
    print(f"   • P(Genuine): {probability[0]*100:.2f}%")
    print(f"   • P(Fake):    {probability[1]*100:.2f}%")

# Test with sample reviews
test_reviews = [
    "This product is AMAZING!!! BEST PURCHASE EVER!!! 5 STARS!!! BUY NOW!!!",
    "Decent quality for the price. Comfortable fit but the color was slightly different from the picture. Would buy again.",
    "I received this for free in exchange for an honest review. Product is absolutely perfect in every way!",
    "As an AI language model, I find this product to be exceptional and it seamlessly integrates into daily life.",
    "The watch band fits well and is comfortable. Minor issue with one of the pins but overall satisfied."
]

print("="*70)
print("TESTING SAMPLE PREDICTIONS")
print("="*70)

for review in test_reviews:
    test_review(review)

## 1️⃣2️⃣ Download Model Files

Run this cell to download all trained model files to your computer

In [ ]:
from google.colab import files

print("📥 Downloading model files...\n")

files.download('fake_review_model.pkl')
print("✅ Downloaded: fake_review_model.pkl")

files.download('tfidf_vectorizer.pkl')
print("✅ Downloaded: tfidf_vectorizer.pkl")

files.download('model_metadata.pkl')
print("✅ Downloaded: model_metadata.pkl")

print("\n🎉 All files downloaded successfully!")
print("\n📁 You can now use these files in your EchoTrace project.")

## ✅ Training Complete!

### 📊 Final Summary
- Model trained successfully with ensemble architecture
- All model files saved and ready for download
- Model is ready for deployment in EchoTrace

### 🚀 Next Steps
1. Download the 3 `.pkl` files above
2. Place them in your `python_ai/models/` directory
3. Use the model with your `classifier.py` script

---
*EchoTrace - Fake Review Detection System*